# Data Preparation - Olist Marketplace
Daten strukturiert und reproduzierbar verarbeiten

## Verbindung mit Duckdb

In [34]:
import duckdb
from pathlib import Path

# Verbindung zur DuckDB (in-memory reicht völlig)
con = duckdb.connect()

# Hilfsfunktion für SQL-Abfragen
def sql(q):
    return con.sql(q).df()

# Projektpfad bestimmen (Notebook liegt in /notebooks)
DATA = Path("../data/raw/brazilian-ecommerce")

In [35]:
# Alle CSV-Dateien in DuckDB registrieren
for f in DATA.glob("*.csv"):
    name = f.stem.replace("olist_", "").replace("_dataset", "")
    
    con.sql(f"""
        CREATE OR REPLACE TABLE {name} AS
        SELECT * FROM read_csv_auto('{f.as_posix()}')
        """
           )
           
# Alle Tabellen anzeigen
sql("SHOW TABLES")

,name
0,customers
1,geolocation
2,order_items
3,order_payments
4,order_reviews
5,orders
6,product_category_name_translation
7,products
8,sellers


## Erstellung der Dataframes pro Kernaufgabe

### EDA für erste Kernaufgabe

In [28]:
# Dataframe für Aufgabe 1. für EDA
df_rfm_eda = sql("""
SELECT 
    c.customer_id,
    c.customer_unique_id,
    c.customer_city,
    c.customer_state,
    c.customer_zip_code_prefix,
    o.order_id,
    o.order_status,
    o.order_purchase_timestamp,
    o.order_approved_at,
    op.payment_value,
    op.payment_type,
    op.payment_installments,
    oi.price,
    oi.freight_value,
    p.product_id,
    pcnt.product_category_name_english,
    
FROM customers AS c
JOIN orders o 
        ON c.customer_id = o.customer_id
JOIN order_payments op 
        ON o.order_id = op.order_id
JOIN order_items oi 
        ON o.order_id = oi.order_id
JOIN products p 
        ON oi.product_id = p.product_id
JOIN product_category_name_translation pcnt 
        ON pcnt.product_category_name = p.product_category_name
    """)

In [29]:
df_rfm_eda

,customer_id,customer_unique_id,customer_city,customer_state,customer_zip_code_prefix,order_id,order_status,order_purchase_timestamp,order_approved_at,payment_value,payment_type,payment_installments,price,freight_value,product_id,product_category_name_english
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,franca,SP,14409,00e7ee1b050b8499577073aeb2a297a1,delivered,2017-05-16 15:05:35,2017-05-16 15:22:12,146.87,credit_card,2,124.99,21.88,a9516a079e37a9c9c36b9b78b10169e8,office_furniture
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,sao bernardo do campo,SP,09790,29150127e6685892b6eab3eec79f59c7,delivered,2018-01-12 20:48:24,2018-01-12 20:58:32,335.48,credit_card,8,289.00,46.48,4aa6014eceb682077f9dc4bffebc05b0,housewares
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,sao paulo,SP,01151,b2059ed67ce144a36e2aa97d2c9e9ad2,delivered,2018-05-19 16:07:45,2018-05-20 16:19:10,157.73,credit_card,7,139.94,17.79,bd07b66896d6f1494f5b86251848ced7,office_furniture
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,mogi das cruzes,SP,08775,951670f92359f4fe4a63112aa7306eba,delivered,2018-03-13 16:06:38,2018-03-13 17:29:19,173.30,credit_card,1,149.94,23.36,a5647c44af977b148e0a3a4751a09e2e,office_furniture
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,campinas,SP,13056,6b7d50bd145f6fc7f33cebabd7e49d0f,delivered,2018-07-29 09:51:30,2018-07-29 10:10:09,252.25,credit_card,8,230.00,22.25,9391a573abe00141c56e38d84d7d5b3b,home_confort
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
115873,b7c889215de76857c7967c1011125d2d,522e244a96d13876c5bac4985a8d5075,curitiba,PR,82410,1a9543c90f188e2e4fb14327ad4a9c9b,delivered,2018-01-30 15:28:21,2018-01-31 15:30:30,662.34,credit_card,6,79.99,30.40,e8316a4667e5870c85e906b1f062bde1,office_furniture
115874,55a52a925384404b4e57b82938bf6062,0fff4016f4008007ba2bb7d63ebae39d,mogi das cruzes,SP,08780,eb6b32a82d2459d0ce7f2089f9eba0f2,delivered,2018-04-29 21:49:07,2018-05-01 03:15:51,12.31,voucher,1,99.90,21.78,78efe838c04bbc568be034082200ac20,furniture_decor
115875,55a52a925384404b4e57b82938bf6062,0fff4016f4008007ba2bb7d63ebae39d,mogi das cruzes,SP,08780,eb6b32a82d2459d0ce7f2089f9eba0f2,delivered,2018-04-29 21:49:07,2018-05-01 03:15:51,11.48,voucher,1,99.90,21.78,78efe838c04bbc568be034082200ac20,furniture_decor
115876,55a52a925384404b4e57b82938bf6062,0fff4016f4008007ba2bb7d63ebae39d,mogi das cruzes,SP,08780,eb6b32a82d2459d0ce7f2089f9eba0f2,delivered,2018-04-29 21:49:07,2018-05-01 03:15:51,11.80,voucher,1,99.90,21.78,78efe838c04bbc568be034082200ac20,furniture_decor


In [30]:
df_rfm_eda.describe()

,order_purchase_timestamp,order_approved_at,payment_value,payment_installments,price,freight_value
count,115878,115864,115878.000000,115878.000000,115878.000000,115878.000000
mean,2017-12-31 10:21:34.849333,2017-12-31 21:47:15.546899,173.016646,2.945572,120.924716,20.075868
min,2016-09-04 21:15:19,2016-10-04 09:43:32,0.000000,0.000000,0.850000,0.000000
25%,2017-09-12 16:02:01.250000,2017-09-13 02:12:04.750000,61.010000,1.000000,39.900000,13.080000
50%,2018-01-19 12:46:34,2018-01-19 20:16:40.500000,108.200000,2.000000,74.900000,16.320000
75%,2018-05-04 19:09:41.750000,2018-05-05 03:33:40,189.720000,4.000000,134.900000,21.220000
max,2018-09-03 09:06:57,2018-09-03 17:40:06,13664.080000,24.000000,6735.000000,409.680000
std,NaN,NaN,268.110998,2.779978,184.217144,15.870900


In [31]:
df_rfm_eda.dtypes

customer_id                              object
customer_unique_id                       object
customer_city                            object
customer_state                           object
customer_zip_code_prefix                 object
order_id                                 object
order_status                             object
order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
payment_value                           float64
payment_type                             object
payment_installments                      int64
price                                   float64
freight_value                           float64
product_id                               object
product_category_name_english            object
dtype: object

In [32]:
df_rfm_eda.isna().sum()

customer_id                       0
customer_unique_id                0
customer_city                     0
customer_state                    0
customer_zip_code_prefix          0
order_id                          0
order_status                      0
order_purchase_timestamp          0
order_approved_at                14
payment_value                     0
payment_type                      0
payment_installments              0
price                             0
freight_value                     0
product_id                        0
product_category_name_english     0
dtype: int64

In [ ]:
df_rfm_eda[df_rfm_eda.isna().any(axis=1)].head(14
                                               )

,customer_id,customer_unique_id,customer_city,customer_state,customer_zip_code_prefix,order_id,order_status,order_purchase_timestamp,order_approved_at,payment_value,payment_type,payment_installments,price,freight_value,product_id,product_category_name_english
10343,0bf35cac6cc7327065da879e2d90fae8,c4c0011e639bdbcf26059ddc38bd3c18,varzea paulista,SP,13225,d77031d6a3c8a52f019764e68f211c69,delivered,2017-02-18 11:04:19,NaT,39.95,boleto,1,28.99,10.96,02a79d79e818ad0be36cfc843a6af7ad,sports_leisure
15866,1e101e0daffaddce8159d25a8e53f2b2,c8822fce1d0bfa7ddf0da24fff947172,macae,RJ,27945,12a95a3c06dbaec84bcfb0e2da5d228a,delivered,2017-02-17 13:05:55,NaT,95.76,boleto,1,79.99,15.77,c6dd917a0be2a704582055949915ab32,cool_stuff
24290,d5de688c321096d15508faae67a27051,d49f3dae6bad25d05160fc17aca5942d,conselheiro lafaiete,MG,36400,7002a78c79c519ac54022d4f8a65e6e8,delivered,2017-01-19 22:26:59,NaT,60.42,boleto,1,45.90,14.52,c3b271f47e73d0c9ccf1b43b7606c705,furniture_decor
26450,07a2a7e0f63fd8cb757ed77d4245623c,79af1bbf230a2630487975aa5d7d6220,paraisopolis,MG,37660,51eb2eebd5d76a24625b31c33dd41449,delivered,2017-02-18 15:52:27,NaT,77.06,boleto,1,59.90,17.16,7868a64aa111bbb4f41f8e1146c0becb,furniture_decor
31845,68d081753ad4fe22fc4d410a9eb1ca01,2e0a2166aa23da2472c6a60c4af6f7a6,sao paulo,SP,03573,d69e5d356402adc8cf17e08b5033acfb,delivered,2017-02-19 01:28:47,NaT,163.43,boleto,1,149.80,13.63,cae2e38942c8489d9d7a87a3f525c06b,furniture_decor
45751,d85919cb3c0529589c6fa617f5f43281,c094ac95fcd52f821809ec232a7a6956,sao vendelino,RS,95795,3c0b8706b065f9919d0505d3b3343881,delivered,2017-02-17 15:53:27,NaT,157.19,boleto,1,133.99,23.20,db8ed3d08891d16a2438a67ab3acb740,bed_bath_table
48287,74bebaf46603f9340e3b50c6b086f992,f79be7c08dd24b72d34634f1b89333a4,sao jose de ribamar,MA,65110,2babbb4b15e6d2dfe95e2de765c97bce,delivered,2017-02-18 17:15:03,NaT,106.81,boleto,1,79.99,26.82,c6dd917a0be2a704582055949915ab32,cool_stuff
51136,684cb238dc5b5d6366244e0e0776b450,6ff8b0d7b35d5c945633b8d60165691b,santos,SP,11030,c1d4211b3dae76144deccd6c74144a88,delivered,2017-01-19 12:48:08,NaT,54.51,boleto,1,39.99,14.52,5ab02ca028398131a5ae91401eb49788,sports_leisure
61317,a3d3c38e58b9d2dfb9207cab690b6310,5a4fa4919cbf2b049e72be460a380e5b,abaete,MG,35620,2eecb0d85f281280f79fa00f9cec1a95,delivered,2017-02-17 17:21:55,NaT,154.23,boleto,1,135.00,19.23,4fd676d9c4723d475026e40aeae56957,garden_tools
69833,2127dc6603ac33544953ef05ec155771,8a9a08c7ca8900a200d83cf838a07e0b,cotia,SP,06708,e04abd8149ef81b95221e88f6ed9ab6a,delivered,2017-02-18 14:40:00,NaT,349.01,boleto,1,309.90,39.11,0e20a07ca1714df21f9b07ca3bf7c682,small_appliances


In [ ]:
df_rfm = sql("""
SELECT 
    c.customer_id,
    c.customer_unique_id,
    c.customer_city,
    c.customer_state,
    c.customer_zip_code_prefix,
    o.order_id,
    o.order_purchase_timestamp,
    o.order_approved_at,
    op.payment_value,
    op.payment_type,
    op.payment_installments,
    oi.price,
    oi.freight_value,
    p.product_id,
    pcnt.product_category_name_english,
    
FROM customers AS c
LEFT JOIN orders o 
        ON c.customer_id = o.customer_id
LEFT JOIN order_payments op 
        ON o.order_id = op.order_id
LEFT JOIN order_items oi 
        ON o.order_id = oi.order_id
LEFT JOIN products p 
        ON oi.product_id = p.product_id
LEFT JOIN product_category_name_translation pcnt 
        ON pcnt.product_category_name = p.product_category_name
    """)

In [ ]:

# Dataframe mit allen wichtigen Spalten
sql("""
SELECT 
    c.customer_unique_id,
    COUNT(o.order_id) as frequency,
    MAX(o.order_approved_at) as recency,
    SUM(p.payment_value) as monetary
FROM customers c
LEFT JOIN orders o ON c.customer_id = o.customer_id
LEFT JOIN order_payments p ON o.order_id = p.order_id
GROUP BY c.customer_unique_id;
    """)


,customer_unique_id,frequency,recency,monetary
0,d0ff1a7468fcc46b8fc658ab35d2a12c,1,2017-11-21 00:14:22,29.75
1,92fd8aa5948e20c43a014c44c025c5e1,1,2018-02-17 16:15:34,106.95
2,48d4bf53229ec530405b4e461ae6adb0,1,2017-10-19 15:35:35,95.52
3,49eb55b407f86a1e87d7898594e65c90,1,2017-05-23 03:50:25,180.45
4,e4000306cf2f63714e6bb70dd20a6592,3,2017-06-08 21:30:18,71.14
...,...,...,...,...
96091,e0ca3ec6d4a9c866e92f30c1b23310e5,1,2017-05-09 08:22:06,141.63
96092,1f76f5c883109a343cca6b54c0303144,1,2017-09-06 10:38:14,371.18
96093,71205b9daf992360e033d1d37517d02d,1,2017-04-04 22:30:19,167.59
96094,92e30f5f39b91d3056988747521a9789,1,2018-03-01 22:10:49,194.12


### EDA für zweite Kernaufgabe

In [41]:
# Dataframe für Aufgabe 2. für EDA
df_pc_eda = sql("""
SELECT 
    c.customer_unique_id,
    o.order_id,
    o.order_status,
    o.order_purchase_timestamp,
    o.order_approved_at,
    op.payment_value,
    oi.price,
    oi.freight_value,
    p.product_id,
    pcnt.product_category_name_english,
    r.review_score
    
FROM customers AS c
JOIN orders o 
        ON c.customer_id = o.customer_id
JOIN order_payments op 
        ON o.order_id = op.order_id
JOIN order_items oi 
        ON o.order_id = oi.order_id
JOIN products p 
        ON oi.product_id = p.product_id
JOIN product_category_name_translation pcnt 
        ON pcnt.product_category_name = p.product_category_name
JOIN order_reviews r ON o.order_id = r.order_id
    """)

In [42]:
df_pc_eda

,customer_unique_id,order_id,order_status,order_purchase_timestamp,order_approved_at,payment_value,price,freight_value,product_id,product_category_name_english,review_score
0,871766c5855e863f6eccc05f988b23cb,00010242fe8c5a6d1ba2dd792cb16214,delivered,2017-09-13 08:59:02,2017-09-13 09:45:35,72.19,58.90,13.29,4244733e06e7ecb4970a6e2683c13e61,cool_stuff,5
1,eb28e67c4c0b83846050ddfb8a35d051,00018f77f2f0320c557190d7a144bdd3,delivered,2017-04-26 10:53:06,2017-04-26 11:05:13,259.83,239.90,19.93,e5f2d52b802189ee658865ca93d83a8f,pet_shop,4
2,3818d81c6709e39d06b2738a8d3a2474,000229ec398224ef6ca0657da4fc703e,delivered,2018-01-14 14:33:31,2018-01-14 14:48:30,216.87,199.00,17.87,c777355d18b72b67abbeef9df44fd0fd,furniture_decor,5
3,af861d436cfc08b2c2ddefd0ba074622,00024acbcdf0a6daa1e931b038114c75,delivered,2018-08-08 10:00:35,2018-08-08 10:10:18,25.78,12.99,12.79,7634da152a4610f1595efa32f14722fc,perfumery,4
4,64b576fb70d441e8f1b2d7d446e483c5,00042b26cf59d7ce69dfabb4e55b4fd9,delivered,2017-02-04 13:57:51,2017-02-04 14:10:13,218.04,199.90,18.14,ac6c3623068f30de03045865e4e10089,garden_tools,5
...,...,...,...,...,...,...,...,...,...,...,...
115604,0c9aeda10a71f369396d0c04dce13a64,fffc94f6ce00a00581880bf54a75a037,delivered,2018-04-23 13:57:06,2018-04-25 04:11:01,343.40,299.99,43.41,4aa6014eceb682077f9dc4bffebc05b0,housewares,5
115605,0da9fe112eae0c74d3ba1fe16de0988b,fffcd46ef2263f404302a634eb57f7eb,delivered,2018-07-14 10:26:46,2018-07-17 04:31:48,386.53,350.00,36.53,32e07fd915822b0765e448c4dd74c828,computers_accessories,5
115606,cd79b407828f02fdbba457111c38e4c4,fffce4705a9662cd70adb13d4a31832d,delivered,2017-10-23 17:07:56,2017-10-24 17:14:25,116.85,99.90,16.95,72a30483855e2eafc67aee5dc2560482,sports_leisure,5
115607,eb803377c9315b564bdedad672039306,fffe18544ffabc95dfada21779c9644f,delivered,2017-08-14 23:02:59,2017-08-15 00:04:32,64.71,55.99,8.72,9c422a519119dcad7575db5af1ba540e,computers_accessories,5


In [43]:
df_pc_eda.describe()

,order_purchase_timestamp,order_approved_at,payment_value,price,freight_value,review_score
count,115609,115595,115609.000000,115609.000000,115609.000000,115609.000000
mean,2017-12-31 04:27:50.933335,2017-12-31 15:53:50.673195,172.387379,120.619850,20.056880,4.034409
min,2016-09-04 21:15:19,2016-10-04 09:43:32,0.000000,0.850000,0.000000,1.000000
25%,2017-09-12 11:14:11,2017-09-12 18:04:35.500000,60.870000,39.900000,13.080000,4.000000
50%,2018-01-19 03:30:43,2018-01-19 14:57:12,108.050000,74.900000,16.320000,5.000000
75%,2018-05-04 15:56:31,2018-05-05 02:13:51,189.480000,134.900000,21.210000,5.000000
max,2018-09-03 09:06:57,2018-09-03 17:40:06,13664.080000,6735.000000,409.680000,5.000000
std,NaN,NaN,265.873969,182.653476,15.836184,1.385584


In [44]:
df_pc_eda.dtypes

customer_unique_id                       object
order_id                                 object
order_status                             object
order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
payment_value                           float64
price                                   float64
freight_value                           float64
product_id                               object
product_category_name_english            object
review_score                              int64
dtype: object

In [45]:
df_pc_eda.isna().sum()

customer_unique_id                0
order_id                          0
order_status                      0
order_purchase_timestamp          0
order_approved_at                14
payment_value                     0
price                             0
freight_value                     0
product_id                        0
product_category_name_english     0
review_score                      0
dtype: int64

### EDA für dritte Kernaufgabe

In [47]:
# Dataframe für Aufgabe 2. für EDA
df_service_eda = sql("""
SELECT 
    o.order_id,
    o.order_status,
    o.order_purchase_timestamp,
    o.order_approved_at,
    o.order_delivered_carrier_date,
    o.order_delivered_customer_date,
    o.order_estimated_delivery_date,
    r.review_score,
    s.seller_id,
    s.seller_city,
    s.seller_state,
    c.customer_city,
    c.customer_state,              
    pcnt.product_category_name_english,
     
FROM orders o
JOIN order_reviews r ON o.order_id = r.order_id
JOIN order_items oi 
        ON o.order_id = oi.order_id
JOIN sellers s
        ON oi.seller_id = s.seller_id
JOIN customers c 
        ON c.customer_id = o.customer_id
JOIN products p 
        ON oi.product_id = p.product_id
JOIN product_category_name_translation pcnt 
        ON pcnt.product_category_name = p.product_category_name
    """)

In [48]:
df_service_eda

,order_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,review_score,seller_id,seller_city,seller_state,customer_city,customer_state,product_category_name_english
0,00e7ee1b050b8499577073aeb2a297a1,delivered,2017-05-16 15:05:35,2017-05-16 15:22:12,2017-05-23 10:47:57,2017-05-25 10:35:35,2017-06-05,4,7c67e1448b00f6e969d365cea6b010ab,itaquaquecetuba,SP,franca,SP,office_furniture
1,29150127e6685892b6eab3eec79f59c7,delivered,2018-01-12 20:48:24,2018-01-12 20:58:32,2018-01-15 17:14:59,2018-01-29 12:41:19,2018-02-06,5,b8bc237ba3788b23da09c0f1f3a3288c,itajai,SC,sao bernardo do campo,SP,housewares
2,b2059ed67ce144a36e2aa97d2c9e9ad2,delivered,2018-05-19 16:07:45,2018-05-20 16:19:10,2018-06-11 14:31:00,2018-06-14 17:58:51,2018-06-13,5,7c67e1448b00f6e969d365cea6b010ab,itaquaquecetuba,SP,sao paulo,SP,office_furniture
3,951670f92359f4fe4a63112aa7306eba,delivered,2018-03-13 16:06:38,2018-03-13 17:29:19,2018-03-27 23:22:42,2018-03-28 16:04:25,2018-04-10,5,7c67e1448b00f6e969d365cea6b010ab,itaquaquecetuba,SP,mogi das cruzes,SP,office_furniture
4,6b7d50bd145f6fc7f33cebabd7e49d0f,delivered,2018-07-29 09:51:30,2018-07-29 10:10:09,2018-07-30 15:16:00,2018-08-09 20:55:48,2018-08-15,5,4a3ca9315b744ce9f8e9374361493884,ibitinga,SP,campinas,SP,home_confort
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
110745,1a9543c90f188e2e4fb14327ad4a9c9b,delivered,2018-01-30 15:28:21,2018-01-31 15:30:30,2018-02-16 16:28:33,2018-03-16 20:03:53,2018-03-13,1,7c67e1448b00f6e969d365cea6b010ab,itaquaquecetuba,SP,curitiba,PR,office_furniture
110746,808c7c69c2778bdf4689eee0286e2bef,canceled,2018-02-22 07:57:07,2018-02-22 08:10:27,NaT,NaT,2018-03-13,1,8a32e327fe2c1b3511609d81aaf9f042,sao paulo,SP,sao paulo,SP,furniture_decor
110747,3304c0c857a9c77a201a551f5a3cacb8,delivered,2017-05-04 12:57:21,2017-05-04 13:10:46,2017-05-05 11:50:12,2017-05-08 10:27:31,2017-05-25,5,ddd51ae8cda92f3995a51fc0f0f3eec7,rio de janeiro,RJ,rio de janeiro,RJ,housewares
110748,cb1f3a44e8b8527e16913306a4d3de2f,delivered,2018-08-07 09:03:02,2018-08-08 09:05:09,2018-08-08 15:01:00,2018-08-15 19:28:29,2018-08-24,4,53243585a1d6dc2643021fd1853d8905,lauro de freitas,BA,porto alegre,RS,telephony


In [49]:
df_service_eda.describe()


,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,review_score
count,110750,110736,109605,108457,110750,110750.000000
mean,2018-01-01 12:44:37.691458,2018-01-02 00:18:39.936452,2018-01-05 14:22:58.117586,2018-01-15 00:21:00.601399,2018-01-25 08:59:40.301580,4.035395
min,2016-09-04 21:15:19,2016-09-15 12:16:38,2016-10-08 10:34:01,2016-10-11 13:46:32,2016-10-04 00:00:00,1.000000
25%,2017-09-14 08:05:20,2017-09-14 13:30:22,2017-09-18 22:27:28,2017-09-26 23:18:38,2017-10-05 00:00:00,4.000000
50%,2018-01-20 22:39:33,2018-01-22 13:53:44,2018-01-24 23:32:35,2018-02-03 18:38:41,2018-02-16 00:00:00,5.000000
75%,2018-05-05 15:14:23.750000,2018-05-05 22:13:43,2018-05-08 15:07:00,2018-05-16 12:53:08,2018-05-28 00:00:00,5.000000
max,2018-09-03 09:06:57,2018-09-03 17:40:06,2018-09-11 19:48:28,2018-10-17 13:22:46,2018-10-25 00:00:00,5.000000
std,NaN,NaN,NaN,NaN,NaN,1.385325


In [50]:
df_service_eda.dtypes


order_id                                 object
order_status                             object
order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
review_score                              int64
seller_id                                object
seller_city                              object
seller_state                             object
customer_city                            object
customer_state                           object
product_category_name_english            object
dtype: object

In [51]:
df_service_eda.isna().sum()

order_id                            0
order_status                        0
order_purchase_timestamp            0
order_approved_at                  14
order_delivered_carrier_date     1145
order_delivered_customer_date    2293
order_estimated_delivery_date       0
review_score                        0
seller_id                           0
seller_city                         0
seller_state                        0
customer_city                       0
customer_state                      0
product_category_name_english       0
dtype: int64